# Check 2: Initial state and recording integrity

This notebook verifies that the collector saves the initial state, every subsequent timestep, and every fish attribute. This is an implementation check, rather than a result inherited from a paper.

We compare the saved records with snapshots taken directly from the live fish after each step. This is stronger than checking row counts alone: 101 rows could still contain a duplicate, skip the initial state, or record the wrong attributes.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "shiner_model.py").is_file()
)
MODEL_DIR = PROJECT_ROOT
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

import numpy as np
import pandas as pd
from shiner_model import GoldenShinersModel, compute_group_metrics

TOLERANCE = 1e-12


## Capture the initial state and each completed step

The initial snapshot is taken immediately after model construction, before calling `step()`. A second snapshot is taken after each of 100 steps. `Step` is the discrete update index; elapsed simulated time is `Step × dt`.

The individual records must contain `X`, `Y`, `DirX`, `DirY`, and `Speed` for the same 20 fish at every step.


In [2]:
def fish_snapshot(model):
    rows = [
        {
            "AgentID": fish.unique_id,
            "X": float(fish.position[0]),
            "Y": float(fish.position[1]),
            "DirX": float(fish.direction[0]),
            "DirY": float(fish.direction[1]),
            "Speed": float(fish.speed),
        }
        for fish in model.agents
    ]
    return pd.DataFrame(rows).set_index("AgentID").sort_index()


recording_model = GoldenShinersModel(n_fish=20, seed=2026)
initial_snapshot = fish_snapshot(recording_model)
initial_group_metrics = compute_group_metrics(recording_model)
initial_model_records = recording_model.datacollector.get_model_vars_dataframe().copy(deep=True)
initial_agent_records = recording_model.datacollector.get_agent_vars_dataframe().copy(deep=True)

assert recording_model.steps == 0
assert len(initial_model_records) == 1
assert len(initial_agent_records) == 20
pd.testing.assert_frame_equal(initial_agent_records.xs(0, level="Step"), initial_snapshot)

snapshots = {0: initial_snapshot}
metric_snapshots = {0: initial_group_metrics}
for step in range(1, 101):
    recording_model.step()
    assert recording_model.steps == step
    snapshots[step] = fish_snapshot(recording_model)
    metric_snapshots[step] = compute_group_metrics(recording_model)

model_records = recording_model.datacollector.get_model_vars_dataframe()
agent_records = recording_model.datacollector.get_agent_vars_dataframe()

pd.DataFrame([
    {"Quantity": "Initial group records", "Observed": len(initial_model_records), "Expected": 1},
    {"Quantity": "Initial individual records", "Observed": len(initial_agent_records), "Expected": 20},
    {"Quantity": "Final group records", "Observed": len(model_records), "Expected": 101},
    {"Quantity": "Final individual records", "Observed": len(agent_records), "Expected": 2020},
    {"Quantity": "First recorded step", "Observed": int(model_records.index.min()), "Expected": 0},
    {"Quantity": "Last recorded step", "Observed": int(model_records.index.max()), "Expected": 100},
])


,Quantity,Observed,Expected
0,Initial group records,1,1
1,Initial individual records,20,20
2,Final group records,101,101
3,Final individual records,2020,2020
4,First recorded step,0,0
5,Last recorded step,100,100


**Interpretation.** The collector contains the initial state plus 100 completed updates: 101 group records and `101 × 20 = 2020` individual records. The first index is 0 and the last is 100. The individual attributes at step 0 match the live initial fish, so the first row is genuinely the initial state.


## Compare every saved record with its live snapshot

The checks below verify all steps and all fish, not just the displayed first and last rows. They also check that earlier records are not changed when the fish move later. Group records are compared with metrics recomputed at the same completed timestep.


In [3]:
expected_steps = list(range(101))
expected_ids = set(initial_snapshot.index)
expected_columns = ["X", "Y", "DirX", "DirY", "Speed"]

assert list(model_records.index) == expected_steps
assert list(agent_records.columns) == expected_columns
assert agent_records.index.names == ["Step", "AgentID"]
assert agent_records.index.is_unique
assert agent_records.groupby(level="Step").size().eq(20).all()
assert not model_records.isna().any().any()
assert not agent_records.isna().any().any()
assert np.isfinite(model_records.select_dtypes(include="number").to_numpy()).all()
assert np.isfinite(agent_records.to_numpy()).all()

max_agent_error = 0.0
for step in expected_steps:
    saved = agent_records.xs(step, level="Step").sort_index()
    assert set(saved.index) == expected_ids
    pd.testing.assert_frame_equal(saved, snapshots[step], check_exact=True)
    max_agent_error = max(max_agent_error, np.abs(saved.to_numpy() - snapshots[step].to_numpy()).max())

expected_group_records = pd.DataFrame.from_dict(metric_snapshots, orient="index")
pd.testing.assert_frame_equal(
    model_records, expected_group_records,
    check_exact=False, atol=TOLERANCE, rtol=TOLERANCE,
)
pd.testing.assert_frame_equal(model_records.iloc[:1], initial_model_records)
pd.testing.assert_frame_equal(agent_records.xs(0, level="Step"), initial_snapshot)

numeric_columns = model_records.select_dtypes(include="number").columns
max_group_error = np.abs(
    model_records[numeric_columns].to_numpy()
    - expected_group_records[numeric_columns].to_numpy()
).max()

recording_checks = pd.DataFrame([
    {"Check": "Exactly one group record per step, including t=0", "Result": "passed"},
    {"Check": "Exactly one record per fish and step", "Result": "passed"},
    {"Check": "All five individual attributes match live snapshots", "Result": "passed"},
    {"Check": "All group metrics match the corresponding completed state", "Result": "passed"},
    {"Check": "Earlier records remain unchanged", "Result": "passed"},
    {"Check": "No missing, infinite, duplicate, or skipped records", "Result": "passed"},
])
print(f"Maximum individual-record error: {max_agent_error:.3g}")
print(f"Maximum group-record error: {max_group_error:.3g}")
print(f"Elapsed simulated time: {recording_model.steps * recording_model.dt:g} time units")
recording_checks


Maximum individual-record error: 0
Maximum group-record error: 0
Elapsed simulated time: 12.5 time units


,Check,Result
0,"Exactly one group record per step, including t=0",passed
1,Exactly one record per fish and step,passed
2,All five individual attributes match live snap...,passed
3,All group metrics match the corresponding comp...,passed
4,Earlier records remain unchanged,passed
5,"No missing, infinite, duplicate, or skipped re...",passed


**Interpretation.** Every stored individual value matches the corresponding live snapshot, and every stored group metric belongs to that same timestep. The initial records are still unchanged after 100 updates. This verifies the ordering `decide all → advance all → record`, the attribute mapping, and the history stored by the collector.


## Check that observation does not change movement

Two models start with the same seed and parameters. One records normally; the other discards calls to `DataCollector.collect()` after the initial state. We compare positions, directions and speeds after every update.

If merely collecting data changed a random draw or updated a fish, these trajectories could diverge. This comparison checks that recording remains an observer of the social model.


In [4]:
observed_model = GoldenShinersModel(n_fish=20, seed=17)
unrecorded_model = GoldenShinersModel(n_fish=20, seed=17)
unrecorded_model.datacollector.collect = lambda current_model: None

pd.testing.assert_frame_equal(fish_snapshot(observed_model), fish_snapshot(unrecorded_model))
max_observer_difference = 0.0
for _ in range(100):
    observed_model.step()
    unrecorded_model.step()
    observed = fish_snapshot(observed_model)
    unrecorded = fish_snapshot(unrecorded_model)
    pd.testing.assert_frame_equal(observed, unrecorded, check_exact=True)
    max_observer_difference = max(
        max_observer_difference,
        np.abs(observed.to_numpy() - unrecorded.to_numpy()).max(),
    )

assert len(observed_model.datacollector.get_model_vars_dataframe()) == 101
assert len(unrecorded_model.datacollector.get_model_vars_dataframe()) == 1

pd.DataFrame([
    {"Quantity": "Normally recorded group states", "Observed": 101},
    {"Quantity": "Discarded-recording group states (initial state only)", "Observed": 1},
    {"Quantity": "Maximum difference between fish trajectories", "Observed": max_observer_difference},
    {"Quantity": "Compared transitions", "Observed": 100},
])


,Quantity,Observed
0,Normally recorded group states,101.0
1,Discarded-recording group states (initial stat...,1.0
2,Maximum difference between fish trajectories,0.0
3,Compared transitions,100.0


**Interpretation.** Discarding the observations leaves the fish trajectories identical. The collector is not responsible for creating alignment, cohesion, or rotation in these runs.

**Outcome of check 2.** The implementation stores `t=0`, every subsequent timestep, and all five individual attributes correctly for the tested run. These checks establish recording integrity; the formulas themselves are tested separately in `03_metric_validation.ipynb`, and collective dynamics are assessed across seeds in the later checks.
